# Sensoriamento Inteligente — Acessibilidade às UBS em Campina Grande (PB)

**Disciplina: Sensoriamento Inteligente e Dados Geoespaciais - 2026.2**

**Professores: Francisco Dantas Nobre Neto e Tiago Brasileiro Araújo**

**Alunos: Pedro Ivo Santos Leite, Lucas Matheus Santos da Silva**

**Notebook de projeto final | Dados vetoriais reais | Python / Google Colab**

**Objetivo:** integrar a malha de setores censitários do IBGE (2022), os agregados populacionais do Censo 2022 e pontos georreferenciados de UBS, calcular indicadores de acessibilidade, aplicar **K-Means**, informações das UBS e produzir mapas e arquivos exportáveis.

**Fontes:** [malha IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/), [agregados básicos por setor](https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Setor_csv/), [CNES / Ministério da Saúde](https://dadosabertos.saude.gov.br/dataset/cnes-cadastro-nacional-de-estabelecimentos-de-saude), [OpenStreetMap](https://www.openstreetmap.org/), DATASUS.

> **Transparência:** a execução automática coleta as unidades no OpenStreetMap/Overpass (dados reais colaborativos, não equivalentes ao CNES). Para usar pontos oficiais do CNES, ative a importação opcional de um CSV validado. Nenhum resultado é pré-calculado ou inventado. A execução requer internet e depende da disponibilidade dos servidores públicos.

> **Limite:** distância euclidiana em metros entre pontos representativos de setores e UBS, não distância em rede nem tempo de deslocamento. A contagem de unidades cadastradas não comprova atendimento disponível.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Instalar bibliotecas e configurar parâmetros

In [ ]:
%pip -q install geopandas pyogrio shapely scikit-learn folium requests matplotlib openpyxl

from pathlib import Path
import io, zipfile, re, time, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import folium
from folium.features import GeoJsonTooltip
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.neighbors import BallTree

MUNICIPIO = "2504009"           # Campina Grande / PB
EPSG_METRICO = 31985           # SIRGAS 2000 / UTM 25S
RAIO_M = 2000                  # parâmetro experimental, não limite oficial
K_CANDIDATOS = [2, 3, 4]
FONTE_UBS = "OSM"               # "OSM" ou "CSV_CNES"
CAMINHO_CNES_CSV = "/content/ubs_cnes_campina_grande.csv"
PASTA = Path('/content/resultados_ubs')
PASTA.mkdir(parents=True, exist_ok=True)

URL_MALHA = ('https://geoftp.ibge.gov.br/organizacao_do_territorio/'
             'malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/'
             'censo_2022/setores/shp/UF/PB_setores_CD2022.zip')
URL_CENSO = ('https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/'
             'Agregados_por_Setores_Censitarios/Agregados_por_Setor_csv/'
             'Agregados_por_setores_basico_BR_20260520.zip')
print('Configuração pronta. Fonte de UBS:', FONTE_UBS)

## 2. Baixar malha oficial e filtrar Campina Grande

A malha é baixada diretamente do FTP geográfico do IBGE. O filtro utiliza o código municipal, não um recorte aproximado por coordenadas.

In [ ]:
def baixar_zip(url, destino):
    destino = Path(destino)
    if destino.exists() and destino.stat().st_size > 1000:
        print('Em cache:', destino.name)
        return destino
    print('Baixando:', url)
    with requests.get(url, timeout=(25, 240), stream=True,
                      headers={'User-Agent': 'ProjetoAcademicoSensoriamento/1.0'}) as resposta:
        resposta.raise_for_status()
        with open(destino, 'wb') as f:
            for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                if bloco: f.write(bloco)
    if not zipfile.is_zipfile(destino):
        destino.unlink(missing_ok=True)
        raise ValueError('Download não é um ZIP válido; confira a fonte oficial.')
    print('OK:', round(destino.stat().st_size / 1e6, 1), 'MB')
    return destino

malha_zip = baixar_zip(URL_MALHA, PASTA / 'malha_pb_2022.zip')
with zipfile.ZipFile(malha_zip) as z:
    shp = next((p for p in z.namelist() if p.lower().endswith('.shp')), None)
    if shp is None: raise ValueError('ZIP da malha não contém arquivo SHP.')
    z.extractall(PASTA / 'malha_pb')
shp_path = PASTA / 'malha_pb' / shp
malha = gpd.read_file(shp_path)
print('Colunas da malha:', malha.columns.tolist())
assert 'CD_MUN' in malha.columns and 'CD_SETOR' in malha.columns
setores = malha.loc[malha['CD_MUN'].astype(str).str.strip() == MUNICIPIO].copy()
if setores.empty: raise ValueError('Nenhum setor localizado; confira código e versão da malha.')
setores['CD_SETOR'] = setores['CD_SETOR'].astype(str).str.strip()
setores = setores.to_crs(EPSG_METRICO)
setores = setores.loc[setores.geometry.notna() & ~setores.geometry.is_empty].copy()
setores['area_km2'] = setores.geometry.area / 1e6
print(f'Setores municipais: {len(setores)} | CRS métrico: {setores.crs}')
display(setores[['CD_SETOR','area_km2']].head())

## 3. Importar população oficial do Censo 2022

A variável **V0001** do arquivo de agregados básicos corresponde ao **total de pessoas**. O arquivo nacional é lido em blocos para economizar memória. Setores ausentes ou com valor inválido são preservados como faltantes, **nunca substituídos por população artificial**.

In [ ]:
censo_zip = baixar_zip(URL_CENSO, PASTA / 'censo_2022_basico.zip')
with zipfile.ZipFile(censo_zip) as z:
    arquivos = [p for p in z.namelist() if p.lower().endswith('.csv') and not p.startswith('__MACOSX')]
    print('CSV(s) encontrados:', arquivos[:10])
    if not arquivos: raise ValueError('Não há CSV dentro do ZIP do Censo.')
    # A publicação básica pode conter mais de um CSV; escolher aquele com CD_SETOR e V0001
    popula = None
    for arquivo in arquivos:
        with z.open(arquivo) as fh:
            leitor = pd.read_csv(fh, sep=';', dtype=str, encoding='iso-8859-1',
                                  chunksize=100_000, low_memory=False)
            cols = [str(c).strip().upper() for c in leitor._engine.names]
            if 'CD_SETOR' not in cols or 'V0001' not in cols:
                continue
            col_cod = leitor._engine.names[cols.index('CD_SETOR')]
            col_pop = leitor._engine.names[cols.index('V0001')]
            partes = []
            for parte in leitor:
                filtro = parte[col_cod].astype(str).str.strip().str.startswith(MUNICIPIO)
                if filtro.any():
                    partes.append(parte.loc[filtro, [col_cod, col_pop]].copy())
            if partes:
                popula = pd.concat(partes, ignore_index=True)
                popula.columns = ['CD_SETOR','populacao']
                print('Arquivo utilizado:', arquivo)
                break
if popula is None or popula.empty:
    raise ValueError('População não encontrada no ZIP: revise colunas/encoding/layout do IBGE.')
popula['CD_SETOR'] = popula['CD_SETOR'].astype(str).str.strip()
popula['populacao'] = pd.to_numeric(popula['populacao'].astype(str).str.replace(',','.', regex=False), errors='coerce')
popula = popula.drop_duplicates(subset=['CD_SETOR'])
setores = setores.merge(popula, on='CD_SETOR', how='left', validate='one_to_one')
setores['densidade_hab_km2'] = setores['populacao'] / setores['area_km2'].replace(0, np.nan)
print('Setores sem população:', setores['populacao'].isna().sum(), 'de', len(setores))
print('Total de pessoas em setores com dado:', int(setores['populacao'].sum()))
display(setores[['CD_SETOR','populacao','area_km2','densidade_hab_km2']].head())

## 4. Obter pontos reais das UBS

**Padrão:** buscar feições de saúde do OpenStreetMap cuja denominação indique **UBS, USF, Unidade Básica de Saúde ou Unidade de Saúde da Família**. O filtro por nome é intencionalmente conservador: pode deixar de fora UBS sem nome ou com nomes genéricos. **Não representa um cadastro oficial completo.**

**Alternativa CNES:** altere `FONTE_UBS = "CSV_CNES"` na primeira célula e envie um CSV **previamente filtrado e conferido como UBS de Campina Grande** contendo `nome`, `latitude`, `longitude` (graus decimais WGS84). Assim o fluxo usa pontos de uma extração oficial validada, sem depender dos nomes de campos mutáveis do CNES.

# Não precisa executar célula se o csv já estiver sido gerado.

In [ ]:
%%script false

# Ler arquivo do DATASUS baixado (tbEstabelecimento202610.csv)

from google.colab import files
import pandas as pd
import io

# Solicita o upload do arquivo CSV local
print("Selecione o seu arquivo .csv:")
uploaded = files.upload()

# Identifica o nome do arquivo enviado e carrega no pandas
if uploaded:
    nome_arquivo = list(uploaded.keys())[0]
    try:
        # Tenta ler com separador ';' e codificação 'latin1' (comum em dados do Datasus/CNES)
        df_usuario = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]), sep=';', encoding='latin1')
        print(f"\nArquivo '{nome_arquivo}' importado com sucesso usando sep=';' e encoding='latin1'!")
    except Exception as e:
        # Fallback caso a tentativa anterior falhe
        print(f"Falha na primeira tentativa de leitura: {e}\nTentando com configurações padrão...")
        df_usuario = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]), sep=',', encoding='utf-8', errors='replace')
        print(f"\nArquivo '{nome_arquivo}' importado com tratamento de erros de encoding!")

    display(df_usuario.head())
else:
    print("Nenhum arquivo foi selecionado.")

In [ ]:
# 1. Atualizar a fonte de dados global do notebook para ler o CSV gerado do CNES (2026)

FONTE_UBS = 'CSV_CNES'
CAMINHO_CNES_CSV = '/content/ubs_cnes_campina_grande_202610.csv'

print(f"Configuração atualizada! Fonte de UBS definida para: {FONTE_UBS}")
print(f"Caminho do arquivo de dados: {CAMINHO_CNES_CSV}")

In [ ]:
# 1. Visualizar as colunas disponíveis para identificar os campos de município, nome e coordenadas
# 2.Célula comentada/desativada nativamente para evitar CalledProcessError do %%script

if 'df_usuario' in globals():
    print("Colunas do CNES importado:", list(df_usuario.columns))

    # Procura colunas típicas do CNES para latitude, longitude, município e nome
    col_lat = [c for c in df_usuario.columns if 'lat' in c.lower()]
    col_lon = [c for c in df_usuario.columns if 'lon' in c.lower()]
    col_mun = [c for c in df_usuario.columns if 'mun' in c.lower() or 'co_ibge' in c.lower()]

    print(f"\nColunas sugeridas:")
    print(f"- Latitude: {col_lat}")
    print(f"- Longitude: {col_lon}")
    print(f"- Município: {col_mun}")

    # Filtro específico para Campina Grande (Código IBGE: 250400)
    if 'CO_MUNICIPIO_GESTOR' in df_usuario.columns:
        df_cg = df_usuario[df_usuario['CO_MUNICIPIO_GESTOR'].astype(str).str.startswith('250400')].copy()
    elif 'CO_IBGE' in df_usuario.columns:
        df_cg = df_usuario[df_usuario['CO_IBGE'].astype(str).str.startswith('250400')].copy()
    else:
        col_municipio = col_mun[0] if col_mun else 'CO_MUNICIPIO_GESTOR'
        df_cg = df_usuario[df_usuario[col_municipio].astype(str).str.contains('250400')].copy()

    print(f"\nRegistros encontrados para Campina Grande: {len(df_cg)}")

    if not df_cg.empty:
        nome_col = 'NO_FANTASIA' if 'NO_FANTASIA' in df_cg.columns else ('NO_RAZAO_SOCIAL' if 'NO_RAZAO_SOCIAL' in df_cg.columns else df_cg.columns[5])
        lat_col = 'NU_LATITUDE' if 'NU_LATITUDE' in df_cg.columns else (col_lat[0] if col_lat else 'latitude')
        lon_col = 'NU_LONGITUDE' if 'NU_LONGITUDE' in df_cg.columns else (col_lon[0] if col_lon else 'longitude')

        df_ubs_final = pd.DataFrame()
        df_ubs_final['nome'] = df_cg[nome_col]
        df_ubs_final['latitude'] = pd.to_numeric(df_cg[lat_col], errors='coerce')
        df_ubs_final['longitude'] = pd.to_numeric(df_cg[lon_col], errors='coerce')

        df_ubs_final = df_ubs_final.dropna(subset=['latitude', 'longitude'])
        df_ubs_final = df_ubs_final[(df_ubs_final['latitude'] != 0) & (df_ubs_final['longitude'] != 0)]

        df_ubs_final.to_csv('/content/ubs_cnes_campina_grande.csv', index=False, encoding='utf-8')
        print(f"\nSucesso! {len(df_ubs_final)} UBS georreferenciadas salvas em '/content/ubs_cnes_campina_grande.csv'")
        display(df_ubs_final.head())
    else:
        print("Não foi possível isolar os dados de Campina Grande. Verifique os nomes das colunas de município.")
else:
    print("Objeto df_usuario não carregado em memória. Pule esta etapa caso já possua o arquivo CSV final.")

In [ ]:
# Filtrando os estabelecimentos do CNES pelo código do município de Campina Grande (250400)
# Geralmente no CNES a coluna é 'CO_MUNICIPIO_GESTOR'

col_municipio = 'CO_MUNICIPIO_GESTOR' if 'CO_MUNICIPIO_GESTOR' in df_usuario.columns else 'CO_IBGE'

# Realiza o filtro
df_campina = df_usuario[df_usuario[col_municipio].astype(str).str.startswith('250400')].copy()

print(f"Foram encontrados {len(df_campina)} registros para o município de Campina Grande (250400/2504009).")

# Exibe as principais colunas para análise
colunas_principais = ['CO_CNES', 'NO_FANTASIA', 'CO_MUNICIPIO_GESTOR', 'NU_LATITUDE', 'NU_LONGITUDE']
colunas_existentes = [c for c in colunas_principais if c in df_campina.columns]

display(df_campina[colunas_existentes].head(20))

In [ ]:
# 1. Identificar colunas de classificação de tipo de estabelecimento

col_tipo = [c for c in df_campina.columns if 'tipo' in c.lower() or 'tp_est' in c.lower() or 'co_est' in c.lower()]
print("Colunas de tipo encontradas:", col_tipo)

# Usaremos a coluna 'TP_ESTABELECIMENTO' ou similar se existir, caso contrário usaremos busca por texto no nome
termo_busca = r'\b(UBS|USF|PSF)\b|UNIDADE\s+B[AÁ]SICA|SA[UÚ]DE\s+DA\s+FAM[IÍ]LIA|POSTO\s+DE\s+SA[UÚ]DE'

# Filtro textual (Nome Fantasia)
filtro_nome = df_campina['NO_FANTASIA'].astype(str).str.contains(termo_busca, case=False, na=False)

# Filtro por Código de Tipo de Estabelecimento (Posto de Saúde = 01, Centro de Saúde/UBS = 02)
# Ajustamos dinamicamente conforme o nome real da coluna no seu CSV
col_tp_est = next((c for c in col_tipo if 'tp_est' in c.lower() or 'co_tipo' in c.lower()), None)

if col_tp_est:
    # Garante que os códigos sejam tratados como strings padronizadas com 2 dígitos (ex: '01', '02')
    df_campina[col_tp_est] = df_campina[col_tp_est].astype(str).str.zfill(2)
    filtro_tipo = df_campina[col_tp_est].isin(['01', '02'])
    df_ubs_filtrado = df_campina[filtro_nome | filtro_tipo].copy()
    print(f"Filtro aplicado usando Nome e Tipo de Estabelecimento ({col_tp_est}).")
else:
    df_ubs_filtrado = df_campina[filtro_nome].copy()
    print("Filtro aplicado baseado exclusivamente nas palavras-chave do Nome Fantasia.")

print(f"Total de UBS encontradas em Campina Grande: {len(df_ubs_filtrado)}")

# 2. Criar e salvar a tabela final formatada para o projeto de acessibilidade
df_final_cnes = pd.DataFrame()
df_final_cnes['nome'] = df_ubs_filtrado['NO_FANTASIA']
df_final_cnes['latitude'] = pd.to_numeric(df_ubs_filtrado['NU_LATITUDE'], errors='coerce')
df_final_cnes['longitude'] = pd.to_numeric(df_ubs_filtrado['NU_LONGITUDE'], errors='coerce')

# Remover coordenadas nulas ou zeradas
df_final_cnes = df_final_cnes.dropna(subset=['latitude', 'longitude'])
df_final_cnes = df_final_cnes[(df_final_cnes['latitude'] != 0) & (df_final_cnes['longitude'] != 0)]

# Salvar no caminho esperado do projeto
df_final_cnes.to_csv('/content/ubs_cnes_campina_grande_202610.csv', index=False, encoding='utf-8')
print(f"Arquivo salvo com sucesso em '/content/ubs_cnes_campina_grande_202610.csv' ({len(df_final_cnes)} registros válidos com coordenadas).")

display(df_final_cnes.head(15))

In [ ]:
# 2. Carregar as 91 UBS como GeoDataFrame e reprojetar para o CRS métrico (EPSG 31985)

import pandas as pd
import geopandas as gpd
from pathlib import Path
import zipfile
import requests

# Fallback caso o arquivo com sufixo _2026 não tenha sido gerado ainda
caminho_alvo = Path(CAMINHO_CNES_CSV)
if not caminho_alvo.exists():
    caminho_alvo = Path('/content/ubs_cnes_campina_grande_202610.csv')

tabela_ubs_cnes = pd.read_csv(caminho_alvo)

# Converter coordenadas brutas em pontos geométricos (WGS 84)
ubs = gpd.GeoDataFrame(
    tabela_ubs_cnes,
    geometry=gpd.points_from_xy(tabela_ubs_cnes['longitude'], tabela_ubs_cnes['latitude']),
    crs=4326
)

# Reprojetar para o CRS métrico para cálculo exato de distâncias em metros
ubs = ubs.to_crs(EPSG_METRICO)

# Garantir que a variável 'setores' esteja carregada na memória
try:
    setores
except NameError:
    print("Variável 'setores' não encontrada em memória. Reconstruindo e baixando malha oficial do IBGE...")
    pasta_malha = Path('/content/resultados_ubs/malha_pb')
    pasta_malha.mkdir(parents=True, exist_ok=True)
    caminho_zip = Path('/content/resultados_ubs/malha_pb_2022.zip')

    # Download caso o zip não esteja no disco
    if not caminho_zip.exists():
        print('Baixando malha PB do IBGE...')
        r = requests.get(URL_MALHA, timeout=120)
        caminho_zip.write_bytes(r.content)

    with zipfile.ZipFile(caminho_zip) as z:
        z.extractall(pasta_malha)

    shp_path = next(pasta_malha.glob('**/*.shp'))
    malha_completa = gpd.read_file(shp_path)
    setores = malha_completa.loc[malha_completa['CD_MUN'].astype(str).str.strip() == MUNICIPIO].copy()
    setores['CD_SETOR'] = setores['CD_SETOR'].astype(str).str.strip()
    setores = setores.to_crs(EPSG_METRICO)
    setores = setores.loc[setores.geometry.notna() & ~setores.geometry.is_empty].copy()
    setores['area_km2'] = setores.geometry.area / 1e6

# Filtrar espacialmente apenas os pontos contidos no limite de Campina Grande
limite_mun = setores.geometry.union_all()
ubs = ubs.loc[ubs.geometry.intersects(limite_mun)].copy()

print(f"UBS do CNES carregadas e validadas com sucesso!")
print(f"Total de UBS espacialmente válidas dentro de Campina Grande: {len(ubs)}")
display(ubs.head(10))


In [ ]:
# Exportando o arquivo final com o nome solicitado pelo usuário

caminho_saida_2026 = '/content/ubs_cnes_campina_grande_202610.csv'

# Selecionando apenas as colunas desejadas e salvando
df_final_cnes[['nome', 'latitude', 'longitude']].to_csv(caminho_saida_2026, index=False, encoding='utf-8')

print(f"Arquivo exportado com sucesso: {caminho_saida_2026}")
print(f"Total de linhas exportadas: {len(df_final_cnes)}")


## 5. Indicadores de distância, cobertura experimental e índice de prioridade

Cada setor é representado por um ponto interno (`representative_point`). A distância até a UBS mais próxima é euclidiana, no CRS métrico. O índice **IPS = 0,6 × distância normalizada + 0,4 × população normalizada** é um critério didático e **não é política oficial de saúde**.

In [ ]:
# 1. Garantir que a coluna 'populacao' e as densidades estejam presentes na variável 'setores'

if 'populacao' not in setores.columns and 'popula' in globals():
    print("Recuperando dados de população e recalculando densidades...")
    setores = setores.merge(popula, on='CD_SETOR', how='left', validate='one_to_one')
    setores['densidade_hab_km2'] = setores['populacao'] / setores['area_km2'].replace(0, np.nan)

# Geometrias em EPSG métrico e pontos representativos internos.
pontos = setores.geometry.representative_point()
xy_setores = np.column_stack((pontos.x.to_numpy(), pontos.y.to_numpy()))
xy_ubs = np.column_stack((ubs.geometry.x.to_numpy(), ubs.geometry.y.to_numpy()))
from scipy.spatial import cKDTree
arvore = cKDTree(xy_ubs)
distancias, ids = arvore.query(xy_setores, k=1)
setores['distancia_ubs_m'] = distancias
setores['ubs_mais_proxima'] = ubs.iloc[ids]['nome'].to_numpy()
setores['ubs_2km'] = [len(v) for v in arvore.query_ball_point(xy_setores, r=RAIO_M)]

def minmax_serie(s):
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi):
        return pd.Series(np.nan,index=s.index)
    if hi == lo:
        return pd.Series(0.0,index=s.index)
    return (s-lo)/(hi-lo)

setores['IPS'] = (0.6 * minmax_serie(setores['distancia_ubs_m']) +
                  0.4 * minmax_serie(setores['populacao'].clip(lower=0)))
setores['distancia_maior_2km'] = setores['distancia_ubs_m'] > RAIO_M
print('Distância mediana (m):', round(setores['distancia_ubs_m'].median(),1))
print('Setores > 2 km:', int(setores['distancia_maior_2km'].sum()))
validos_pop = setores.dropna(subset=['populacao'])
if validos_pop['populacao'].sum() > 0:
    percentual = 100 * validos_pop.loc[validos_pop.distancia_maior_2km,'populacao'].sum() / validos_pop.populacao.sum()
    print(f'População em setores cujo ponto representativo está >2 km: {percentual:.1f}% (dos setores com população conhecida)')
else:
    print('Sem população válida; percentual populacional não calculável.')
display(setores[['CD_SETOR','populacao','distancia_ubs_m','ubs_2km','IPS']].head())

## 6. Aprendizado não supervisionado: K-Means

A IA agrupa setores usando distância à UBS, densidade populacional e número de UBS no raio de 2 km. **Clusters não possuem ordem natural**: a classificação em prioridade menor/intermediária/maior será feita *após* o ajuste, segundo a média do IPS de cada cluster. O IPS não entra na matriz de treinamento.

In [ ]:
atributos = ['distancia_ubs_m','densidade_hab_km2','ubs_2km']
dados = setores.loc[setores[atributos + ['populacao','IPS']].notna().all(axis=1)].copy()
dados = dados.loc[(dados['area_km2'] > 0) & (dados['populacao'] >= 0)].copy()
if len(dados) < 8: raise ValueError('São necessários ao menos 8 setores com dados completos.')
scaler = StandardScaler()
X = scaler.fit_transform(dados[atributos])
resultados = []
for k in K_CANDIDATOS:
    if len(dados) <= k: continue
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    rotulos = km.fit_predict(X)
    if len(set(rotulos)) < 2: continue
    resultados.append({'k':k, 'silhouette':silhouette_score(X,rotulos),
                       'davies_bouldin':davies_bouldin_score(X,rotulos),
                       'inercia':km.inertia_})
aval = pd.DataFrame(resultados)
if aval.empty: raise ValueError('Não foi possível obter clusters distintos; revise atributos.')
display(aval.round(3))
melhor_k = int(aval.sort_values(['silhouette','davies_bouldin'], ascending=[False,True]).iloc[0]['k'])
modelo = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
dados['cluster'] = modelo.fit_predict(X)
medias_ips = dados.groupby('cluster')['IPS'].mean().sort_values()
nomes = ['Prioridade relativa menor', 'Prioridade relativa intermediária', 'Prioridade relativa maior'] if melhor_k == 3 else [f'Faixa relativa {i+1}/{melhor_k}' for i in range(melhor_k)]
rotular = dict(zip(medias_ips.index,nomes))
dados['classe_relativa'] = dados['cluster'].map(rotular)
print('K selecionado por Silhouette:',melhor_k)
display(dados.groupby(['cluster','classe_relativa'])[atributos + ['IPS','populacao']].mean().round(2))

## 7. Métricas das UBS encontradas

In [ ]:
# 1. Calcular os índices Silhoutte e  Davies-Bouldin

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].plot(aval['k'],aval['silhouette'],marker='o')
axes[0].set(title='Silhouette por número de clusters',xlabel='K',ylabel='Silhouette')
axes[0].grid(alpha=.2)
axes[1].plot(aval['k'],aval['davies_bouldin'],marker='o')
axes[1].set(title='Davies-Bouldin por número de clusters',xlabel='K',ylabel='Davies-Bouldin (menor é melhor)')
axes[1].grid(alpha=.2)
plt.tight_layout(); plt.savefig(PASTA/'metricas_clusters.png',dpi=160);plt.show()

resumo = dados.groupby('classe_relativa').agg(
    setores=('CD_SETOR','count'),
    populacao=('populacao','sum'),
    distancia_media_m=('distancia_ubs_m','mean'),
    distancia_mediana_m=('distancia_ubs_m','median'),
    IPS_medio=('IPS','mean')).reset_index()
display(resumo.round(2))
resumo.to_csv(PASTA/'resumo_clusters.csv', index=False)
aval.to_csv(PASTA/'metricas_kmeans.csv',index=False)

## 8. Mapas temáticos (arquivos PNG)

In [ ]:
fig, ax = plt.subplots(figsize=(11,9))
setores.plot(column='distancia_ubs_m',cmap='YlOrRd',legend=True,linewidth=0.05,edgecolor='gray',ax=ax)
ubs.plot(ax=ax,color='navy',markersize=13,marker='+',label='UBS mapeada')
ax.set_title('Distância euclidiana à UBS mais próxima — Campina Grande/PB')
ax.set_axis_off();ax.legend();plt.tight_layout()
plt.savefig(PASTA/'mapa_distancia.png',dpi=170,bbox_inches='tight');plt.show()

fig, ax = plt.subplots(figsize=(11,9))
dados.plot(column='cluster',categorical=True,cmap='Set2',legend=True,linewidth=0.05,edgecolor='gray',ax=ax)
ubs.plot(ax=ax,color='black',markersize=11,marker='+',label='UBS mapeada')
ax.set_title(f'Agrupamentos K-Means (K={melhor_k}) — apenas setores com dados completos')
ax.set_axis_off();ax.legend();plt.tight_layout()
plt.savefig(PASTA/'mapa_clusters.png',dpi=170,bbox_inches='tight');plt.show()

## 9. Mapas interativos HTML (Folium), Endereços e Distâncias

In [ ]:
centro = setores.to_crs(4326).geometry.union_all().centroid
mapa = folium.Map(location=[centro.y,centro.x],zoom_start=11,tiles='CartoDB positron')
#mapa = folium.Map(location=[centro.y, centro.x], zoom_start=11, tiles='OpenStreetMap')
mapa_dados = dados[['CD_SETOR','populacao','distancia_ubs_m','ubs_2km','IPS','classe_relativa','geometry']].copy().to_crs(4326)
mapa_dados['distancia_ubs_m'] = mapa_dados['distancia_ubs_m'].round(0)
mapa_dados['IPS'] = mapa_dados['IPS'].round(3)
mapa_dados['populacao'] = mapa_dados['populacao'].astype(int)
cores = ['#2a9d8f','#f4a261','#e76f51','#8d5db5']
ordem = medias_ips.index.tolist()
cor_por_cluster = dict(zip(ordem,cores[:len(ordem)]))
cor_por_classe = {rotular[k]:cor_por_cluster[k] for k in ordem}
folium.GeoJson(mapa_dados.to_json(),name='Setores / K-Means',
    style_function=lambda feat:{'fillColor':cor_por_classe[feat['properties']['classe_relativa']],
                                'color':'#555','weight':0.3,'fillOpacity':0.65},
    tooltip=GeoJsonTooltip(fields=['CD_SETOR','classe_relativa','populacao','distancia_ubs_m','ubs_2km','IPS'],
      aliases=['Setor','Grupo','População','Distância (m)','UBS 2 km','IPS'])).add_to(mapa)
for _, u in ubs.to_crs(4326).iterrows():
    folium.CircleMarker([u.geometry.y,u.geometry.x],radius=4,color='blue',fill=True,
                        tooltip=str(u['nome'])).add_to(mapa)
folium.LayerControl().add_to(mapa)
mapa.save(str(PASTA/'mapa_interativo.html'))
display(mapa)

# Mapa com Linhas de Conexão

Para tornar o mapa interativo ainda mais rico, vamos adicionar linhas tracejadas que conectam o ponto representativo de cada setor censitário à UBS mais próxima, exibindo a distância exata em metros ao passar o mouse.
```

In [ ]:
# 1. Criar um novo mapa focado nas conexões de distância

import folium

centro_coord = setores.to_crs(4326).geometry.union_all().centroid
mapa_distancias_linhas = folium.Map(location=[centro_coord.y, centro_coord.x], zoom_start=12, tiles='CartoDB positron')

# Adicionar os setores de forma sutil
folium.GeoJson(
    mapa_dados.to_json(),
    name='Setores Censitários',
    style_function=lambda x: {'fillColor': '#ececec', 'color': '#aaaaaa', 'weight': 0.5, 'fillOpacity': 0.4}
).add_to(mapa_distancias_linhas)

# Obter as coordenadas dos pontos de origem (setores) e destino (UBS mais próxima) em WGS84
setores_wgs = setores.to_crs(4326)
ubs_wgs = ubs.to_crs(4326)

# Adicionar marcadores para as UBS
for _, u in ubs_wgs.iterrows():
    folium.Marker(
        location=[u.geometry.y, u.geometry.x],
        popup=f"<b>UBS: {u['nome']}</b>",
        icon=folium.Icon(color='blue', icon='plus-sign')
    ).add_to(mapa_distancias_linhas)

# Alterado para [::2] conforme solicitado para visualizar mais conexões (50% de amostragem)
for idx, row in setores_wgs.iloc[::2].iterrows():
    geom_setor = row.geometry.representative_point()
    # Encontrar a coordenada da UBS mais próxima correspondente
    nome_prox = row['ubs_mais_proxima']
    ubs_dest = ubs_wgs[ubs_wgs['nome'] == nome_prox]

    if not ubs_dest.empty:
        geom_ubs = ubs_dest.geometry.values[0]
        dist_m = round(row['distancia_ubs_m'], 1)

        # Desenhar linha conectando o setor à UBS
        folium.PolyLine(
            locations=[[geom_setor.y, geom_setor.x], [geom_ubs.y, geom_ubs.x]],
            color='red',
            weight=1.2,
            opacity=0.7,
            dash_array='5, 5',
            tooltip=f"Distância até {nome_prox}: {dist_m}m"
        ).add_to(mapa_distancias_linhas)

folium.LayerControl().add_to(mapa_distancias_linhas)
mapa_distancias_linhas.save(str(PASTA / 'mapa_distancias_linhas.html'))
display(mapa_distancias_linhas)

# Endereços das UBS

In [ ]:
# 1. Recuperar dados de endereços a partir do DataFrame filtrado do CNES (df_campina)
# 2. Verificamos se as colunas típicas de endereço do CNES estão presentes

print("=== UBS Mapeadas e Endereços (CNES - Campina Grande) ===\n")
rua_col = 'NO_LOGRADOURO' if 'NO_LOGRADOURO' in df_campina.columns else None
num_col = 'NU_ENDERECO' if 'NU_ENDERECO' in df_campina.columns else None
bairro_col = 'NO_BAIRRO' if 'NO_BAIRRO' in df_campina.columns else None

# Exibir os endereços das UBS que estão de fato na nossa lista final georreferenciada
ubs_nomes_validas = set(ubs['nome'].unique())
df_enderecos = df_campina[df_campina['NO_FANTASIA'].isin(ubs_nomes_validas)].drop_duplicates(subset=['NO_FANTASIA'])

for i, (_, row) in enumerate(df_enderecos.iterrows(), 1):
    nome = row['NO_FANTASIA']

    rua = str(row[rua_col]).strip() if rua_col and pd.notna(row[rua_col]) else ""
    numero = str(row[num_col]).strip() if num_col and pd.notna(row[num_col]) else ""
    bairro = str(row[bairro_col]).strip() if bairro_col and pd.notna(row[bairro_col]) else ""

    partes_endereco = []
    if rua and rua.upper() != 'NAN':
        partes_endereco.append(f"{rua}, {numero}" if numero and numero.upper() != 'NAN' else rua)
    if bairro and bairro.upper() != 'NAN':
        partes_endereco.append(bairro)
    partes_endereco.append("Campina Grande")

    endereco_completo = ", ".join(partes_endereco)
    print(f"{i}. Nome: {nome}")
    print(f"   Endereço: {endereco_completo}\n")


## Distâncias entre as Centróides das UBS Mapeadas

Nesta seção, calculamos a distância linear (euclidiana) entre as coordenadas das UBS encontradas utilizando a projeção cartográfica métrica configurada.
```

In [ ]:
# 1. Criar uma matriz de distâncias entre os pontos das UBS

import pandas as pd

nomes_ubs = ubs['nome'].tolist()
geometrias_ubs = ubs.geometry.tolist()

matriz_distancia = []
for i, geom_a in enumerate(geometrias_ubs):
    linha = []
    for j, geom_b in enumerate(geometrias_ubs):
        # Distância em metros convertida para quilômetros
        dist_km = geom_a.distance(geom_b) / 1000.0
        linha.append(round(dist_km, 3))
    matriz_distancia.append(linha)

# Exibir como DataFrame estruturado
df_distancias = pd.DataFrame(matriz_distancia, index=nomes_ubs, columns=nomes_ubs)
print("=== Matriz de Distâncias entre UBS (em km) ===")
display(df_distancias)

## 10. Exportação de dados e geração do pacote de entrega

In [ ]:
# 1. Para evitar conflito case-insensitive ao gravar no GPKG, removemos a coluna 'AREA_KM2' original do IBGE

for df_temp in [setores, dados]:
    if 'AREA_KM2' in df_temp.columns and 'area_km2' in df_temp.columns:
        df_temp.drop(columns=['AREA_KM2'], inplace=True)

# Ajustando tipos de dados para evitar conflito com o driver GPKG/pyogrio
for df_temp in [setores, dados]:
    if 'area_km2' in df_temp.columns:
        df_temp['area_km2'] = df_temp['area_km2'].astype(float)
    if 'populacao' in df_temp.columns:
        df_temp['populacao'] = pd.to_numeric(df_temp['populacao'], errors='coerce')
    if 'densidade_hab_km2' in df_temp.columns:
        df_temp['densidade_hab_km2'] = df_temp['densidade_hab_km2'].astype(float)
    if 'distancia_ubs_m' in df_temp.columns:
        df_temp['distancia_ubs_m'] = df_temp['distancia_ubs_m'].astype(float)
    if 'IPS' in df_temp.columns:
        df_temp['IPS'] = df_temp['IPS'].astype(float)

# Todas as áreas originais com indicadores, inclusive setores sem população,
# junto apenas setores usados na IA e respectivos clusters.

setores.to_file(PASTA/'setores_indicadores.gpkg',layer='setores',driver='GPKG')
dados.to_file(PASTA/'setores_clusters.gpkg',layer='clusters',driver='GPKG')
ubs.to_file(PASTA/'ubs_mapeadas.gpkg',layer='ubs',driver='GPKG')
dados.drop(columns='geometry').sort_values('IPS',ascending=False).to_csv(
    PASTA/'setores_prioritarios.csv', index=False,encoding='utf-8-sig')

nota = f"""Projeto: Acessibilidade geográfica às UBS — Campina Grande/PB
IBGE Censo: 2022; código municipal: {MUNICIPIO}
Data de processamento: {pd.Timestamp.now().isoformat()}
Fonte de pontos UBS: {FONTE_UBS}
Número de pontos válidos: {len(ubs)}
Setores na malha: {len(setores)}
Setores com atributos completos para K-Means: {len(dados)}
CRS de análise: EPSG:{EPSG_METRICO}
Raio experimental: {RAIO_M} metros
K selecionado: {melhor_k}
Limitações: distância em linha reta; ponto representativo por setor;
dados OSM possivelmente incompletos; prioridade relativa não é indicador oficial.
"""
(PASTA/'LEIA_ME.txt').write_text(nota,encoding='utf-8')
zip_final = Path('/content/entrega_sensoriamento_ubs.zip')
with zipfile.ZipFile(zip_final,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for arq in PASTA.iterdir():
        if arq.suffix.lower() in {'.gpkg','.csv','.png','.html','.txt'}:
            z.write(arq,arcname=arq.name)
print(nota)
print('Pacote criado:',zip_final)
try:
    from google.colab import files
    files.download(str(zip_final))
except ImportError:
    print('Fora do Colab: arquivo salvo em',zip_final)

## 11. Discussão e interpretação para o relatório

1. Os setores mais distantes coincidem com os mais populosos? Compare mapas e médias dos grupos.
2. O número de UBS a 2 km diferencia os agrupamentos? Compare médias e dispersão.
3. O valor de **K** selecionado gera grupos espacialmente interpretáveis? Lembre que K-Means não impõe continuidade geográfica.
4. Qual é a fração de setores sem população informada? Como isso pode afetar os resultados?
5. Como mudariam os indicadores se o raio experimental fosse **1 km ou 3 km**?
6. Em uma extensão futura, como ruas, relevo, transporte público e capacidade de atendimento alterariam a análise?

**Referências:** IBGE, Censo Demográfico 2022 / Agregados por Setores Censitários e Malha de Setores; Ministério da Saúde, CNES; OpenStreetMap Contributors, banco de dados geográfico; Pedregosa et al., *Scikit-learn: Machine Learning in Python* (2011).

**Declaração metodológica:** dados censitários referem-se a 2022 e os pontos OSM referem-se ao momento da consulta; a diferença temporal e o caráter colaborativo do cadastro de UBS são limitações. Não conclua que um setor está desassistido apenas por estar longe de uma unidade mapeada.